In [13]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models, regularizers
import seaborn as sns

In [ ]:
#!pip install waymo-open-dataset-tf-2-5-0 --quiet
#from waymo_open_dataset import dataset_pb2 as open_dataset
#load the Waymo dataset
def load_waymo_data(data_dir):
    # Placeholder for loading Waymo data
    # Use the Waymo Open Dataset API to load and preprocess the data
    # dataset = tf.data.TFRecordDataset(filenames)
    # dataset = dataset.map(parse_function)  # Define a parse function to extract features and labels
    # return dataset
    pass

ERROR: Could not find a version that satisfies the requirement waymo-open-dataset-tf-2-5-0 (from versions: none)
ERROR: No matching distribution found for waymo-open-dataset-tf-2-5-0


In [6]:
def extract_features_and_labels(dataset):
    # Placeholder for feature and label extraction
    # You would implement this function to parse the Waymo dataset and extract relevant features and labels
    pass

In [7]:
def transform_data(features):
    # Placeholder for data transformation
    # Implement any necessary transformations such as normalization, encoding, etc.
    pass

In [9]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3,3), padding='same', input_shape=(64, 64, 3)),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        
        tf.keras.layers.Conv2D(64, (3,3), padding='same'),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
        
        tf.keras.layers.Conv2D(128, (3,3), padding='same'),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    ])
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation='sigmoid')
    
    loss = tf.keras.layers.lossess.BinaryCrossentropy()
    optimizer = tf.keras.optimizers.Adam(1e-4)
    
    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
    return model
        
        

In [ ]:
class net(build_model()):
  def __init__(self):
    super(net, self).__init__()
    self.loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    self.optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    self.train_loss = tf.keras.metrics.Mean(name='train_loss')
    self.train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    self.test_loss = tf.keras.metrics.Mean(name='test_loss')
    self.test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')
    self.model = self.net()
    self.model.summary()

In [10]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, filters, stride=1):
        super().__init__()

        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding='same')
        self.bn1 = layers.BatchNormalization()

        self.conv2 = layers.Conv2D(filters, 3, strides=1, padding='same')
        self.bn2 = layers.BatchNormalization()


        if stride != 1:
            self.shortcut = tf.keras.Sequential([
                layers.Conv2D(filters, 1, strides=stride, padding='same'),
                layers.BatchNormalization()
            ])
        else:
            self.shortcut = lambda x: x   # identity

    def call(self, x):
        shortcut = self.shortcut(x)

        x = tf.nn.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))

        x = x + shortcut
        return tf.nn.relu(x)

In [11]:
def build_resnet(input_shape=(224,224,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    # Initial layer
    x = layers.Conv2D(64, 7, strides=2, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # Residual stages
    # Stage 1
    x = ResidualBlock(64, stride=1)(x)
    x = ResidualBlock(64, stride=1)(x)

    # Stage 2
    x = ResidualBlock(128, stride=2)(x)
    x = ResidualBlock(128, stride=1)(x)

    # Stage 3
    x = ResidualBlock(256, stride=2)(x)
    x = ResidualBlock(256, stride=1)(x)

    # Stage 4
    x = ResidualBlock(512, stride=2)(x)
    x = ResidualBlock(512, stride=1)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=regularizers.l2(0.001)
    )(x)

    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model_resnet = models.Model(inputs, outputs)

    return model_resnet

In [16]:
# train_test_split
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train, x_test = x_train/255.0, x_test/255.0

# combine then split 70/30
X = np.concatenate([x_train, x_test], axis=0)
Y = np.concatenate([y_train, y_test], axis=0)

x_train, x_test, y_train, y_test = train_test_split(X, Y, train_size=0.70, random_state=42)

# create a combined dataset after splitting data
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
test_dataset  = tf.data.Dataset.from_tensor_slices((x_test, y_test))

def process_images(image, label):
    #resize to ensure the image fits the model
    image = tf.image.resize(image, (224,224))
    return image, label




train_dataset = (train_dataset
                  .map(process_images)
                  .batch(batch_size=32))
test_dataset = (test_dataset
                  .map(process_images)
                  .batch(batch_size=32))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 52s 0us/step


In [14]:
model_resnet = build_resnet()

model_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_resnet.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 112, 112, 64)   │         9,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_8                │ (None, 56, 56, 64)     │        74,368 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_9                │ (None, 56, 56, 64)     │        74,368 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_10               │ (None, 28, 28, 128)    │       231,296 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_11               │ (None, 28, 28, 128)    │       296,192 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_12               │ (None, 14, 14, 256)    │       921,344 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_13               │ (None, 14, 14, 256)    │     1,182,208 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_14               │ (None, 7, 7, 512)      │     3,677,696 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_15               │ (None, 7, 7, 512)      │     4,723,712 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,257,866 (42.95 MB)

 Trainable params: 11,248,266 (42.91 MB)

 Non-trainable params: 9,600 (37.50 KB)

In [ ]:
history_resnet = model_resnet.fit(train_dataset,  validation_data=test_dataset,
    epochs=10,
    batch_size=32)

Epoch 1/10
 212/1313 ━━━━━━━━━━━━━━━━━━━━ 15:53 866ms/step - accuracy: 0.2322 - loss: 2.5514

In [ ]:
plt.plot(history_resnet.history['accuracy'], label='train')
plt.plot(history_resnet.history['val_accuracy'], label='val')
plt.legend()
plt.title("Accuracy of resnet")
plt.show()